In [1]:
from pyspark.sql import SparkSession

# 1. Gọi quản đốc Spark dậy
spark = SparkSession.builder.appName("Tiki_Flatten").getOrCreate()

# 2. Đọc file Parquet cũ (thay đường dẫn nếu cần)
df = spark.read.parquet("Thanh_Pham_Parquet")

# 3. Chụp X-quang dữ liệu!
df.printSchema()

root
 |-- brand: string (nullable = true)
 |-- id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)



In [2]:
from pyspark.sql.types import *

print("Đang tạo lại cái ba lô Tiki lộn xộn chứa hàng lồng nhau...")

# Tạo dữ liệu giả lập chứa cả Struct (Object) và Array (Danh sách)
data = [
    (1, "Laptop KTC H27T13 2K", {"name": "Tiki Trading", "is_official": True}, ["Freeship 10k", "Trả góp 0%"]),
    (2, "Túi rác Inochi", {"name": "Inochi Official", "is_official": True}, ["Rẻ vô địch", "Mua 8 tặng 1"])
]

# Tự tay vẽ "Bản thiết kế X-quang" (Schema)
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    
    # Đây là cái túi nhỏ (Struct)
    StructField("seller", StructType([               
        StructField("name", StringType(), True),
        StructField("is_official", BooleanType(), True)
    ]), True),
    
    # Đây là hộp kẹo mút (Array)
    StructField("badges", ArrayType(StringType()), True) 
])

df_raw = spark.createDataFrame(data, schema)

# Chụp lại X-quang lần 2!
df_raw.printSchema()

Đang tạo lại cái ba lô Tiki lộn xộn chứa hàng lồng nhau...
root
 |-- id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- seller: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- is_official: boolean (nullable = true)
 |-- badges: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [3]:
from pyspark.sql.functions import col

print("Đang bóc túi seller và sắp xếp lên mặt bàn...")

# Dùng select() kết hợp col() để lấy các trường phẳng và trường lồng sâu
df_phang_mot_nua = df_raw.select(
    col("id"),
    col("product_name"),
    col("seller.name").alias("ten_shop"),             # Thò tay lấy tên shop và đổi tên cột ngay lập tức
    col("seller.is_official").alias("shop_chinh_hang"),
    col("badges")                                     # Tạm thời cứ để nguyên hộp kẹo (Array) badges ở đó
)

# 1. In kết quả ra xem bảng đã phẳng chưa
print("--- BẢNG DỮ LIỆU ĐÃ ĐƯỢC LÀM PHẲNG MỘT NỬA ---")
df_phang_mot_nua.show(truncate=False)

# 2. Chụp lại X-quang xem túi seller đã biến mất chưa
print("--- ẢNH X-QUANG MỚI ---")
df_phang_mot_nua.printSchema()

Đang bóc túi seller và sắp xếp lên mặt bàn...
--- BẢNG DỮ LIỆU ĐÃ ĐƯỢC LÀM PHẲNG MỘT NỬA ---
+---+--------------------+---------------+---------------+--------------------------+
|id |product_name        |ten_shop       |shop_chinh_hang|badges                    |
+---+--------------------+---------------+---------------+--------------------------+
|1  |Laptop KTC H27T13 2K|Tiki Trading   |true           |[Freeship 10k, Trả góp 0%]|
|2  |Túi rác Inochi      |Inochi Official|true           |[Rẻ vô địch, Mua 8 tặng 1]|
+---+--------------------+---------------+---------------+--------------------------+

--- ẢNH X-QUANG MỚI ---
root
 |-- id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- ten_shop: string (nullable = true)
 |-- shop_chinh_hang: boolean (nullable = true)
 |-- badges: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [9]:
from pyspark.sql.functions import col, explode

print("Đang gài bom explode để xé hộp kẹo badges...")

# Dùng explode để bóc mảng thành các dòng độc lập
df_phang_toan_tap = df_phang_mot_nua.select(
    col("id"),
    col("product_name"),
    col("ten_shop"),
    col("shop_chinh_hang"),
    explode(col("badges")).alias("badge_don")  # KÍCH NỔ HỘP KẸO TẠI ĐÂY!
)

# 1. In kết quả cuối cùng
print("--- BẢNG DỮ LIỆU SAU KHI NỔ BOM EXPLODE ---")
df_phang_toan_tap.show(truncate=False)

# 2. Chụp X-quang lần cuối
print("--- ẢNH X-QUANG CUỐI CÙNG ---")
df_phang_toan_tap.printSchema()

Đang gài bom explode để xé hộp kẹo badges...
--- BẢNG DỮ LIỆU SAU KHI NỔ BOM EXPLODE ---
+---+--------------------+---------------+---------------+------------+
|id |product_name        |ten_shop       |shop_chinh_hang|badge_don   |
+---+--------------------+---------------+---------------+------------+
|1  |Laptop KTC H27T13 2K|Tiki Trading   |true           |Freeship 10k|
|1  |Laptop KTC H27T13 2K|Tiki Trading   |true           |Trả góp 0%  |
|2  |Túi rác Inochi      |Inochi Official|true           |Rẻ vô địch  |
|2  |Túi rác Inochi      |Inochi Official|true           |Mua 8 tặng 1|
+---+--------------------+---------------+---------------+------------+

--- ẢNH X-QUANG CUỐI CÙNG ---
root
 |-- id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- ten_shop: string (nullable = true)
 |-- shop_chinh_hang: boolean (nullable = true)
 |-- badge_don: string (nullable = true)



In [5]:
from pyspark.sql.functions import col, explode, explode_outer

print("Đang test rủi ro mất dữ liệu với Bút bi Thiên Long...")

# Tạo dữ liệu test bẫy (Bút bi có mảng rỗng)
data_bay = [
    (1, "Laptop KTC", ["Freeship"]),
    (2, "Bút bi Thiên Long", [])      # <-- Chú ý: Không có badge nào!
]
df_test = spark.createDataFrame(data_bay, ["id", "product_name", "badges"])

print("--- 1. KHI DÙNG EXPLODE THƯỜNG (BẠO LỰC) ---")
# Dùng explode
df_loi = df_test.select(col("id"), col("product_name"), explode(col("badges")).alias("badge"))
df_loi.show()

print("--- 2. KHI DÙNG EXPLODE_OUTER (CÓ BẢO HIỂM) ---")
# Dùng explode_outer
df_an_toan = df_test.select(col("id"), col("product_name"), explode_outer(col("badges")).alias("badge"))
df_an_toan.show()

Đang test rủi ro mất dữ liệu với Bút bi Thiên Long...
--- 1. KHI DÙNG EXPLODE THƯỜNG (BẠO LỰC) ---
+---+------------+--------+
| id|product_name|   badge|
+---+------------+--------+
|  1|  Laptop KTC|Freeship|
+---+------------+--------+

--- 2. KHI DÙNG EXPLODE_OUTER (CÓ BẢO HIỂM) ---
+---+-----------------+--------+
| id|     product_name|   badge|
+---+-----------------+--------+
|  1|       Laptop KTC|Freeship|
|  2|Bút bi Thiên Long|    NULL|
+---+-----------------+--------+



In [7]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, explode_outer

print("Đang tạo mảng chứa cấu trúc phức tạp (Array of Structs)...")

# Dữ liệu mô phỏng: Cuốn sách có 2 danh mục, Bút bi không có danh mục nào
data_phuc_tap = [
    (1, "Sách Đắc Nhân Tâm", [
        {"id": 111, "name": "Sách kỹ năng"}, 
        {"id": 222, "name": "Kỹ năng sống"}
    ]),
    (2, "Bút bi Thiên Long", [])  # Vẫn test mảng rỗng để xem bảo hiểm hoạt động
]

# Vẽ sơ đồ X-quang: Array bọc lấy Struct
schema_phuc_tap = StructType([
    StructField("id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("categories", ArrayType(             # CÁI HỘP
        StructType([                                 # CHỨA NHIỀU CÁI TÚI NHỎ
            StructField("id", IntegerType(), True),
            StructField("name", StringType(), True)
        ])
    ), True)
])

df_cate = spark.createDataFrame(data_phuc_tap, schema_phuc_tap)
print("--- ẢNH X-QUANG ARRAY OF STRUCTS ---")
df_cate.printSchema()

print("--- THỰC HÀNH BÓC TÁCH 2 BƯỚC ---")

# Bước 1: Kích nổ hộp thành nhiều cái túi độc lập
df_step_1 = df_cate.select(
    col("id"), 
    col("product_name"), 
    explode_outer(col("categories")).alias("cate_tui_nho") # Đặt tên mới cho dễ nhớ
)

df_step_1.show(truncate=False)
# Bước 2: Thò tay vào từng cái túi nhỏ lấy đồ
df_step_2 = df_step_1.select(
    col("id"),
    col("product_name"),
    col("cate_tui_nho.id").alias("cate_id"),       # Dấu chấm thần thánh!
    col("cate_tui_nho.name").alias("cate_name")
)

df_step_2.show(truncate=False)


Đang tạo mảng chứa cấu trúc phức tạp (Array of Structs)...
--- ẢNH X-QUANG ARRAY OF STRUCTS ---
root
 |-- id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: integer (nullable = true)
 |    |    |-- name: string (nullable = true)

--- THỰC HÀNH BÓC TÁCH 2 BƯỚC ---
+---+-----------------+-------------------+
|id |product_name     |cate_tui_nho       |
+---+-----------------+-------------------+
|1  |Sách Đắc Nhân Tâm|{111, Sách kỹ năng}|
|1  |Sách Đắc Nhân Tâm|{222, Kỹ năng sống}|
|2  |Bút bi Thiên Long|NULL               |
+---+-----------------+-------------------+

+---+-----------------+-------+------------+
|id |product_name     |cate_id|cate_name   |
+---+-----------------+-------+------------+
|1  |Sách Đắc Nhân Tâm|111    |Sách kỹ năng|
|1  |Sách Đắc Nhân Tâm|222    |Kỹ năng sống|
|2  |Bút bi Thiên Long|NULL   |NULL        |
+---+-----------------+---

In [12]:
from pyspark.sql.functions import col, concat_ws

print("Đang xâu chuỗi các hộp kẹo lại thành 1 hàng...")

data_bay = [
    (1, "Laptop KTC", ["Freeship", "Giảm 5%", "Giảm 10%"]),
    (2, "Bút bi Thiên Long", [])      # <-- Chú ý: Không có badge nào!
]
df_test = spark.createDataFrame(data_bay, ["id", "product_name", "badges"])
# Thay vì dùng explode, ta dùng concat_ws
df_gom_chuoi = df_test.select(
    col("id"),
    col("product_name"),
    # Cú pháp: concat_ws("kí_tự_ngăn_cách", cột_chứa_mảng)
    concat_ws(", ", col("badges")).alias("danh_sach_badge") 
)

df_gom_chuoi.show(truncate=False)

Đang xâu chuỗi các hộp kẹo lại thành 1 hàng...
+---+-----------------+---------------------------+
|id |product_name     |danh_sach_badge            |
+---+-----------------+---------------------------+
|1  |Laptop KTC       |Freeship, Giảm 5%, Giảm 10%|
|2  |Bút bi Thiên Long|                           |
+---+-----------------+---------------------------+



In [13]:
from pyspark.sql.functions import col, when

print("Đang dọn dẹp các ô trống trơn để báo cáo đẹp hơn...")

# Dùng when... otherwise để kiểm tra: NẾU là chuỗi rỗng THÌ điền chữ, NGƯỢC LẠI giữ nguyên
df_sach_se = df_gom_chuoi.withColumn(
    "danh_sach_badge",
    when(col("danh_sach_badge") == "", "Không có khuyến mãi") \
    .otherwise(col("danh_sach_badge"))
)

df_sach_se.show(truncate=False)

Đang dọn dẹp các ô trống trơn để báo cáo đẹp hơn...
+---+-----------------+---------------------------+
|id |product_name     |danh_sach_badge            |
+---+-----------------+---------------------------+
|1  |Laptop KTC       |Freeship, Giảm 5%, Giảm 10%|
|2  |Bút bi Thiên Long|Không có khuyến mãi        |
+---+-----------------+---------------------------+



In [17]:
"""
from pyspark.sql.functions import col, concat_ws, when

print("Đang khởi động Đại Siêu Trình Flatten Data Tiki...")

# Giả định df_thuc_te là dữ liệu cậu đọc từ file Parquet "Thanh_Pham_Parquet"
df_thuc_te = spark.read.parquet("Thanh_Pham_Parquet")

# GOM TOÀN BỘ VÕ CÔNG VÀO 1 PIPELINE
df_final = df_thuc_te.select(
    # 1. Các trường đã phẳng sẵn (Giữ lại những trường tinh túy nhất)
    col("id"),
    col("title").alias("ten_san_pham"),       # Đổi tên cho chuẩn Việt Nam
    col("price").alias("gia_tien"),           # Đảm bảo có giá tiền như mục tiêu
    col("brand").alias("thuong_hieu"),
    
    # 2. Thò tay bóc túi nhỏ (Struct)
    col("seller.name").alias("ten_shop"),
    col("seller.is_official").alias("shop_chinh_hang"),
    
    # 3. Gom chuỗi hộp kẹo (Array) thay vì xé nát bằng explode để bảng báo cáo gọn gàng
    concat_ws(", ", col("badges")).alias("khuyen_mai")
    
).withColumn(
    # 4. Dọn dẹp rác (Null/Empty) ở cột khuyến mãi
    "khuyen_mai",
    when(col("khuyen_mai") == "", "Không có").otherwise(col("khuyen_mai"))
)

print("--- KẾT QUẢ NGHIỆM THU TUẦN 1 ---")
# Phép màu xuất hiện: Hiển thị 20 dòng, không cắt xén chữ (truncate=False)
df_final.show(20, truncate=False)
"""

'\nfrom pyspark.sql.functions import col, concat_ws, when\n\nprint("Đang khởi động Đại Siêu Trình Flatten Data Tiki...")\n\n# Giả định df_thuc_te là dữ liệu cậu đọc từ file Parquet "Thanh_Pham_Parquet"\ndf_thuc_te = spark.read.parquet("Thanh_Pham_Parquet")\n\n# GOM TOÀN BỘ VÕ CÔNG VÀO 1 PIPELINE\ndf_final = df_thuc_te.select(\n    # 1. Các trường đã phẳng sẵn (Giữ lại những trường tinh túy nhất)\n    col("id"),\n    col("title").alias("ten_san_pham"),       # Đổi tên cho chuẩn Việt Nam\n    col("price").alias("gia_tien"),           # Đảm bảo có giá tiền như mục tiêu\n    col("brand").alias("thuong_hieu"),\n    \n    # 2. Thò tay bóc túi nhỏ (Struct)\n    col("seller.name").alias("ten_shop"),\n    col("seller.is_official").alias("shop_chinh_hang"),\n    \n    # 3. Gom chuỗi hộp kẹo (Array) thay vì xé nát bằng explode để bảng báo cáo gọn gàng\n    concat_ws(", ", col("badges")).alias("khuyen_mai")\n    \n).withColumn(\n    # 4. Dọn dẹp rác (Null/Empty) ở cột khuyến mãi\n    "khuyen_mai",

NameError: name 'spark' is not defined